# $2 \times 2$ Dual-Uncertainty RAG Matrix

This notebook demonstrates how open-domain QA (AmbigQA), multi-hop QA (HotpotQA), and scientific claim verification (SciFact) map into the four quadrants:
1. **Q1 (Prompt Clear, Evidence Sufficient) $\rightarrow$ `ANSWER`**
2. **Q2 (Prompt Ambiguous, Evidence Sufficient) $\rightarrow$ `ASK`**
3. **Q3 (Prompt Clear, Evidence Missing/Contradictory) $\rightarrow$ `SEARCH`**
4. **Q4 (Ambiguous/Unverifiable + Unresolvable Corpus) $\rightarrow$ `ABSTAIN`**

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.data_pipeline.schema import QUADRANT_NAMES, ACTION_NAMES
from src.data_pipeline.hf_ingestor import stream_raw_qa_records
from src.data_pipeline.perturbation_engine import DualUncertaintyPerturbationEngine

## 1. Generate All 4 Quadrants from the Same Underlying QA Record
Let's take a single QA record and inspect how the perturbation engine creates the 4 counterfactual RAG states.

In [ ]:
rng = np.random.default_rng(42)
raw_records = stream_raw_qa_records('in_distribution', max_raw_records=10, use_hf_hub=False)
engine = DualUncertaintyPerturbationEngine()

rows = []
for q_id in range(4):
    ex = engine.generate_state(raw_records[0], rng, quadrant=q_id)
    rows.append({
        'Quadrant': QUADRANT_NAMES[q_id],
        'Prompt_Clear': ex.prompt_certainty_label,
        'Evid_Sufficient': ex.evidence_certainty_label,
        'Immediate_Answer_Regret': round(ex.current_answer_regret, 3),
        'Best_Question_VOI': round(float(np.max(ex.question_voi)), 3),
        'Best_Search_VOI': round(float(np.max(ex.rewrite_voi)), 3),
        'Oracle_Optimal_Action': ACTION_NAMES[ex.optimal_macro_action]
    })
pd.DataFrame(rows)

## 2. Why VOI Beats Generic Information Gain (Entropy Reduction) in Quadrant 2 (`ASK`)
In Quadrant 2 (Ambiguous Query + Competing Passages), candidate questions 0..5 probe semantic facets, while questions 6..7 are surface formatting distractors (e.g., *"Bullet points or paragraph?"*). Notice that surface distractors have high **Information Gain** (the user's formatting preference is unknown!), but **negative net VOI** because they do not disambiguate the retrieved passages.

In [ ]:
ex_ask = engine.generate_state(raw_records[0], rng, quadrant=1)
q_df = pd.DataFrame({
    'Candidate_Q_Idx': range(8),
    'Question_Role': [f'Semantic Facet {i}' if i < 6 else 'Formatting Distractor' for i in range(8)],
    'Counterfactual_VOI': np.round(ex_ask.question_voi, 4),
    'Generic_Information_Gain': np.round(ex_ask.question_ig, 3)
}).sort_values('Counterfactual_VOI', ascending=False)
q_df